# FP-Growth Recommender System
This notebook loads data, cleans it, trains FP-Growth, evaluates and builds recommendations.

In [9]:
import pandas as pd
from mlxtend.frequent_patterns import fpgrowth, association_rules
from mlxtend.preprocessing import TransactionEncoder
import pickle
import os

In [12]:
# Load dataset
df = pd.read_excel(r"C:\Users\lenovo\Desktop\FP-Growth Recommender System\data\online_retail_II.xlsx")
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [13]:
# Data cleaning
df = df.dropna(subset=['Customer ID'])
df = df[df['Quantity'] > 0]
df = df[df['Invoice'].astype(str).str.startswith('C') == False]

df['Customer ID'] = df['Customer ID'].astype(str)

df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [14]:
# Create transactions
transactions = df.groupby('Invoice')['Description'].apply(list).tolist()
len(transactions)

19215

In [15]:
# Encode transactions
te = TransactionEncoder()
te_array = te.fit(transactions).transform(transactions)
df_encoded = pd.DataFrame(te_array, columns=te.columns_)
df_encoded.head()

,DOORMAT UNION JACK GUNS AND ROSES,3 STRIPEY MICE FELTCRAFT,4 PURPLE FLOCK DINNER CANDLES,ANIMAL STICKERS,BLACK PIRATE TREASURE CHEST,BROWN PIRATE TREASURE CHEST,Bank Charges,CAMPHOR WOOD PORTOBELLO MUSHROOM,CHERRY BLOSSOM DECORATIVE FLASK,FAIRY CAKE CANDLES,...,ZINC HEART LATTICE CHARGER LARGE,ZINC HEART LATTICE CHARGER SMALL,ZINC HEART LATTICE DOUBLE PLANTER,ZINC HEART LATTICE PLANTER BOWL,ZINC HEART LATTICE T-LIGHT HOLDER,ZINC HEART LATTICE TRAY OVAL,ZINC METAL HEART DECORATION,ZINC POLICE BOX LANTERN,ZINC TOP 2 DOOR WOODEN SHELF,ZINC WILLIE WINKIE CANDLE STICK
0,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


In [17]:
# Train FP-Growth model
frequent_itemsets = fpgrowth(df_encoded, min_support=0.01, use_colnames=True)
frequent_itemsets.head()

,support,itemsets
0,0.069477,frozenset({STRAWBERRY CERAMIC TRINKET BOX})
1,0.018319,frozenset({SAVE THE PLANET MUG})
2,0.016706,frozenset({PINK DOUGHNUT TRINKET POT })
3,0.012126,frozenset({15CM CHRISTMAS GLASS BALL 20 LIGHTS})
4,0.011866,frozenset({PINK CHERRY LIGHTS})


In [18]:
# Generate rules
rules = association_rules(frequent_itemsets, metric="lift", min_threshold=1)
rules = rules.sort_values(by='lift', ascending=False)
rules.head()

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
470,frozenset({CHILDS GARDEN TROWEL PINK}),frozenset({CHILDS GARDEN TROWEL BLUE }),0.012022,0.011553,0.010096,0.839827,72.690418,1.0,0.009957,6.171112,0.998244,0.749035,0.837955,0.856850
471,frozenset({CHILDS GARDEN TROWEL BLUE }),frozenset({CHILDS GARDEN TROWEL PINK}),0.011553,0.012022,0.010096,0.873874,72.690418,1.0,0.009957,7.833255,0.997771,0.749035,0.872339,0.856850
511,frozenset({POPPY'S PLAYHOUSE BEDROOM }),frozenset({POPPY'S PLAYHOUSE LIVINGROOM }),0.014052,0.011918,0.010200,0.725926,60.911208,1.0,0.010033,3.605165,0.997600,0.646865,0.722620,0.790911
510,frozenset({POPPY'S PLAYHOUSE LIVINGROOM }),frozenset({POPPY'S PLAYHOUSE BEDROOM }),0.011918,0.014052,0.010200,0.855895,60.911208,1.0,0.010033,6.841885,0.995446,0.646865,0.853841,0.790911
508,frozenset({POPPY'S PLAYHOUSE LIVINGROOM }),frozenset({POPPY'S PLAYHOUSE KITCHEN}),0.011918,0.015561,0.011033,0.925764,59.493508,1.0,0.010848,13.260976,0.995050,0.670886,0.924591,0.817397


In [ ]:
# Save models
with open('/models/fpgrowth.pkl', 'wb') as f:
    pickle.dump(frequent_itemsets, f)

with open('/models/rules.pkl', 'wb') as f:
    pickle.dump(rules, f)

In [20]:
# Recommendation function
def get_recommendations(basket, rules, top_n=5):
    basket = set(basket)
    recommendations = []

    for _, row in rules.iterrows():
        if row['antecedents'].issubset(basket):
            recommendations.extend(list(row['consequents']))

    return list(set(recommendations))[:top_n]

# Example
get_recommendations(['WHITE HANGING HEART T-LIGHT HOLDER'], rules)

['SWEETHEART CERAMIC TRINKET BOX',
 'VINTAGE UNION JACK BUNTING',
 'WOODEN FRAME ANTIQUE WHITE ',
 'JUMBO BAG RED RETROSPOT',
 'HEART FILIGREE DOVE LARGE']

In [21]:
# Basic evaluation
print("Number of frequent itemsets:", len(frequent_itemsets))
print("Number of rules:", len(rules))

Number of frequent itemsets: 835
Number of rules: 520
